<div dir="rtl">

# Python מינימלי לעבודה עם SQLite

</div>

<div dir="rtl">

## מה צריכה תוכנית Python כדי לשאול מסד נתונים שאלה?

במדריך ה־SQL הקלדתם שאילתות ב־DB Browser. תוכנית שולחת את אותן שאילתות ומקבלת את השורות כערכים של Python. השיעור עובר על ה־Python שתוכנית כזו משתמשת בו, לפי הסדר שבו היא משתמשת בו: ערכים וטקסט, טאפלים ורשימות, הקריאות שמדברות עם מסד הנתונים, פרמטרים, פונקציה שמריצה שאילתה, ורשימה שנבנית מתוצאות של שאילתה.

הנתונים הם טבלת שיתוף האופניים `daily_ridership` של המדריך: 731 ימים אמיתיים, שורה אחת לכל יום. השיעור משתמש בשש מעמודותיה: `year`, `month`, `season`, `weather` (אחת משלוש תוויות), `working_day` (1 ביום עבודה, 0 אחרת) ו־`total_rides` (ההשכרות של אותו יום).

כשיש לנושא שיעור מלא משלו, השיעור הזה חוזר על הרעיון במשפט אחד ומקשר אליו: [ערכים ואוספים](https://dvirross.com/he/courses/intro-data-science/python-values-and-collections), [לולאות ופונקציות](https://dvirross.com/he/courses/intro-data-science/python-loops-and-functions) ו־[SQL מתוך Python](https://dvirross.com/he/courses/intro-data-science/sql-from-python).

</div>

<div dir="rtl">

**איך עובדים:** הריצו את התאים מלמעלה למטה, כי התאים חולקים את המשתנים שלהם. תא שכתוב בו `# Your turn` ריק: כתבו בו את הקוד, הריצו אותו, והריצו את התא שמתחתיו, שבודק את התשובה בלי להראות את הפתרון. במחברת עם הפתרונות יש מתחת לכל משימה את הפתרון, את הטעויות הנפוצות ואת הפלט הצפוי.

</div>

<div dir="rtl">

## הכנה: להביא את קובץ מסד הנתונים

אין צורך בחשבון ואין צורך במפתח. התא הבא מוריד את מסד הנתונים של הקורס מהאתר הציבורי ושומר אותו ליד המחברת (ב־Colab הוא מופיע בחלונית **Files**). אם ההורדה נכשלת, פותחים את הכתובת שלמטה בדפדפן, מורידים את הקובץ ומעלים אותו לחלונית **Files** של Colab.

`https://dvirross.com/materials/courses/intro-data-science/sql-practice.sqlite`

</div>

In [1]:
import os
import urllib.request

DB_URL = 'https://dvirross.com/materials/courses/intro-data-science/sql-practice.sqlite'
DB_PATH = 'sql-practice.sqlite'

if not os.path.exists(DB_PATH):
    # the site refuses the default user agent of Python, so the request names one
    request = urllib.request.Request(DB_URL, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request) as response, open(DB_PATH, 'wb') as file:
        file.write(response.read())
print('קובץ מסד הנתונים מוכן:', DB_PATH)

קובץ מסד הנתונים מוכן: sql-practice.sqlite


<div dir="rtl">

התא הבא מגדיר את הפונקציה `check(task_number)`. אחרי שמריצים את התא של משימה, מריצים את התא שמתחתיו: `check` מריצה שוב את התא שלכם, אוספת את מה שהוא מדפיס ומשווה אותו לפלט הצפוי. היא אומרת אם הפלט נכון, ואם לא באיזו שורה הוא שונה; היא לא מראה את הפתרון.

</div>

In [2]:
import ast, contextlib, hashlib, io

# The expected output of each task, as hashes of its lines (the solutions are not stored here).
EXPECTED = {int(k): v for k, v in {
    '3': [
        '4963af400ece'
    ],
    '4': [
        '7d827bbe5897'
    ],
    '5': [
        '14e4e0edb1cb'
    ],
    '6': [
        '2b855e705c53',
        'fa4ea1bc266c'
    ],
    '7': [
        '28af1299a0b7'
    ]
}.items()}


def check(task):
    """Run the last cell that ran (not a check) again and compare what it prints with the expected output of the task."""
    if task not in EXPECTED:
        print('אין משימה כזו: ', task)
        return
    cells = [s for s in In[:-1] if s.strip() and not s.lstrip().startswith('check(')]   # In: the cells that ran, kept by IPython
    tree = ast.parse(cells[-1] if cells else '')
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None   # a bare last expression counts as printed
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            exec(compile(tree, '<task>', 'exec'), globals())
            if last is not None:
                value = eval(compile(ast.Expression(last.value), '<task>', 'eval'), globals())
                if value is not None:
                    print(repr(value))
    except Exception as error:
        print('התא נכשל בשגיאה:', type(error).__name__ + ':', error)
        return
    got = [hashlib.sha256(line.rstrip().encode()).hexdigest()[:12] for line in out.getvalue().rstrip().splitlines()]
    want = EXPECTED[task]
    if not got:
        print('לא הודפס כלום. הדפיסו את התשובה בתא המשימה, הריצו אותו, והריצו את `check` מיד אחריו.')
    elif len(got) != len(want):
        print('התא הדפיס {got} שורות והמשימה מצפה ל־{want}. קראו שוב את המשימה ואת הרמז.'.format(got=len(got), want=len(want)))
    elif got != want:
        print('שורה {k} שונה מהפלט הצפוי. קראו שוב את המשימה ואת הרמז.'.format(k=next(i for i in range(len(want)) if got[i] != want[i]) + 1))
    else:
        print('OK: הפלט כצפוי.')


<div dir="rtl">

## ערכים, משתנים ו־`print`

**[משתנה](https://dvirross.com/he/courses/intro-data-science/glossary#variable)** (variable) הוא שם שמפנה לערך, כמו המספר 985 או הטקסט `'Mist or cloudy'`. הפונקציה `print` מציגה ערכים על המסך: היא שמה רווח בין ערך לערך, ולכן אפשר להציג בה טקסט ומספרים יחד. שורה שמתחילה ב־`#` היא **הערה** (comment): Python מתעלמת ממנה, והיא כתובה בשביל הקורא.

</div>

In [3]:
# one day of the table
rides = 985
weather = 'Mist or cloudy'
print('Weather:', weather, 'Rides:', rides)

Weather: Mist or cloudy Rides: 985


<div dir="rtl">

## טקסט וגרשיים

טקסט נכתב בין גרשיים, וגרש יחיד וגרשיים כפולים הם אותו דבר. טקסט בין **שלושה גרשים** יכול להימשך על פני כמה שורות. עבור Python, ‏SQL הוא טקסט, והוא מכיל גרשיים יחידים סביב ערכי הטקסט שלו, ולכן כותבים אותו בגרשיים כפולים או בשלושה גרשים:

</div>

In [4]:
label = 'Light rain or snow'
one_line = "SELECT * FROM daily_ridership WHERE year = 2012"
many_lines = """
    SELECT year, month
    FROM daily_ridership
    WHERE weather = 'Mist or cloudy'
"""
print(label)
print(one_line)

Light rain or snow
SELECT * FROM daily_ridership WHERE year = 2012


<div dir="rtl">

## טאפלים, רשימות וציינים

טבלה במסד נתונים היא שורות מתחת לעמודות קבועות. תוצאה של שאילתה מגיעה ל־Python כרשימה של **[טאפלים](https://dvirross.com/he/courses/intro-data-science/glossary#tuple)** (tuple): טאפל נכתב בסוגריים עגולים ומחזיק מספר קבוע של ערכים לפי סדר, אחד לכל עמודה ב־`SELECT`, ואי אפשר לשנות אותו בטעות. **[רשימה](https://dvirross.com/he/courses/intro-data-science/glossary#list)** (list) נכתבת בסוגריים מרובעים ומחזיקה פריטים רבים, כאן שורות רבות. **[ציין](https://dvirross.com/he/courses/intro-data-science/glossary#index)** (index) הוא המקום של פריט, בסוגריים מרובעים, והספירה מתחילה ב־0, ולכן `row[0]` הוא הערך הראשון.

</div>

In [5]:
row = (2011, 1, 985)
rows = [(2011, 1, 985), (2011, 1, 801), (2011, 1, 1349)]
print(row[0], row[2])
print(rows[1])
print(rows[1][2])

2011 985
(2011, 1, 801)
801


<div dir="rtl">

השורה האחרונה לוקחת את השורה השנייה ברשימה, ואז את הערך השלישי באותה שורה. השיעור המלא נמצא ב[טאפלים ורשימות](https://dvirross.com/he/courses/intro-data-science/python-values-and-collections#טאפלים-ורשימות-כמה-ערכים-לפי-סדר).

## התחברות, סמן, הרצה ושליפה

ארבעה שלבים מביאים שורות ממסד נתונים אל Python. כל שלב הוא שורה אחת. השורה `import sqlite3` מאפשרת להשתמש במודול `sqlite3`, שמגיע עם Python.

**התחברות.** הפונקציה `sqlite3.connect` פותחת את קובץ מסד הנתונים שהשם שלו ניתן לה ומחזירה **[חיבור](https://dvirross.com/he/courses/intro-data-science/glossary#connection)** (connection), אובייקט שמייצג את מסד הנתונים הפתוח.

</div>

In [6]:
import sqlite3

conn = sqlite3.connect('sql-practice.sqlite')

<div dir="rtl">

**יצירת סמן.** השיטה `cursor` של החיבור יוצרת **[סמן](https://dvirross.com/he/courses/intro-data-science/glossary#cursor)** (cursor), אובייקט ששולח SQL ומחזיק את התשובה עד שלוקחים אותה.

</div>

In [7]:
c = conn.cursor()

<div dir="rtl">

**הרצת שאילתה.** השיטה `execute` של הסמן שולחת שאילתה אחת. היא מחזירה את הסמן עצמו, ולכן מה שרואים הוא אובייקט הסמן, ושורות התשובה מחכות בו.

</div>

In [8]:
c.execute("""
    SELECT year, month, weather, total_rides
    FROM daily_ridership
    LIMIT 2
""")

<div dir="rtl">

**שליפת התוצאות.** השיטה `fetchall` לוקחת את כל שורות התשובה, כרשימה של טאפלים.

</div>

In [9]:
results = c.fetchall()
print(results)

[(2011, 1, 'Mist or cloudy', 985), (2011, 1, 'Mist or cloudy', 801)]


<div dir="rtl">

השיטה `fetchmany` לוקחת את השורות הבאות, כמספר שנותנים לה. השיטה `fetchone` לוקחת את השורה הבאה לבדה, כטאפל. שאילתה יכולה לבחור 731 שורות בזמן ש־`fetchmany(3)` לוקחת שלוש מהן. מכיוון ש־`execute` מחזירה את הסמן, אפשר להוסיף אחריה שליפה באותה שורה. הערכים של שורה נמצאים לפי הסדר של ה־`SELECT`: העמודה הראשונה שנבחרה היא ציין 0.

</div>

In [10]:
three = c.execute(
    "SELECT year, month, total_rides FROM daily_ridership"
).fetchmany(3)
print(three)
row = c.execute(
    "SELECT month, total_rides FROM daily_ridership LIMIT 1"
).fetchone()
print(row)
print('Month:', row[0])
print('Rides:', row[1])

[(2011, 1, 985), (2011, 1, 801), (2011, 1, 1349)]
(1, 985)
Month: 1
Rides: 985


<div dir="rtl">

## פונקציות סיכום בשאילתה

פונקציות הסיכום של [מדריך ה־SQL](https://dvirross.com/he/courses/intro-data-science/sql-in-db-browser#סיכום-ערכים) פועלות באותו אופן בתוך תוכנית. `SUM(total_rides)` מחברת את ההשכרות של הימים שנבחרו, ו־`COUNT(*)` סופרת את הימים; `COUNT(weather)` סופרת את השורות שיש להן תווית, ו־`COUNT(DISTINCT weather)` סופרת את התוויות השונות. שאילתת סיכום מחזירה שורה אחת, ולכן `fetchone()` היא השליפה המתאימה, והשורה היא טאפל גם כשיש בה ערך אחד.

</div>

In [11]:
result = c.execute("""
    SELECT SUM(total_rides), COUNT(*)
    FROM daily_ridership
    WHERE year = 2011 AND working_day = 0
""").fetchone()
print('Rides:', result[0], 'Days:', result[1])

Rides: 386839 Days: 115


<div dir="rtl">

השאילתה צריכה `SUM` כי כל שורה היא יום עם השכרות רבות: `COUNT(*)` הייתה מחזירה את מספר הימים, 115, ולא את ההשכרות.

### משימה 1 (דוגמה פתורה): ימים ותוויות ב־2012

כמה ימים יש ל־2012 בטבלה, וכמה תוויות מזג אוויר שונות יש להם? השוו בין `COUNT(weather)` ל־`COUNT(DISTINCT weather)`: מה כל אחת סופרת? החליטו לפני שאתם מריצים את הקוד.

</div>

In [12]:
result = c.execute("""
    SELECT COUNT(weather), COUNT(DISTINCT weather)
    FROM daily_ridership
    WHERE year = 2012
""").fetchone()
print('Days:', result[0], 'Different weather labels:', result[1])

Days: 366 Different weather labels: 3


<div dir="rtl">

ל־2012 יש 366 ימים (שנה מעוברת), לכל יום יש תווית, והתוויות הן שלוש בסך הכול. השורה היא טאפל עם שני ערכים.

שאילתה עם `DISTINCT` מחזירה שורה אחת לכל ערך, וכשהיא בוחרת עמודה אחת השורות הן טאפלים עם ערך אחד. Python כותבת טאפל כזה עם פסיק, `('Mist or cloudy',)`. `LIKE '%cloudy%'` מתאים לטקסט שמכיל את `cloudy` בכל מקום.

### משימה 2 (דוגמה פתורה): תוויות שמכילות `cloudy`

כמה תוויות מזג אוויר שונות מכילות את הטקסט `cloudy`? אחר כך הציגו רשימה של התוויות האלה.

</div>

In [13]:
result = c.execute("""
    SELECT COUNT(DISTINCT weather)
    FROM daily_ridership
    WHERE weather LIKE '%cloudy%'
""").fetchone()
print('Labels that contain cloudy:', result[0])
labels = c.execute("""
    SELECT DISTINCT weather
    FROM daily_ridership
    WHERE weather LIKE '%cloudy%'
    LIMIT 10
""").fetchall()
print(labels)

Labels that contain cloudy: 2
[('Mist or cloudy',), ('Clear or partly cloudy',)]


<div dir="rtl">

הספירה היא `result[0]`, והרשימה מחזיקה טאפל עם ערך אחד לכל תווית.

### משימה 3: השכרות בימי עבודה

**נסו בעצמכם.** השתמשו בסמן `c` כדי לקבל את מספר ההשכרות הכולל בימי העבודה של 2012. הדפיסו את הטקסט `Total rides:` ואחריו את המספר, ב־`print` אחד.

</div>

<div dir="rtl">

> **רמז**
>
> כותבים את ה־SQL כמו ב־DB Browser: פונקציית סיכום אחת, ו־`WHERE` עם שני תנאים שמחוברים ב־`AND`. התוצאה היא שורה אחת, ולכן לוקחים אותה עם `fetchone` ומוציאים את המספר מהטאפל בציין.

</div>

<div dir="rtl">

**פתרון**

</div>

In [14]:
result = c.execute("""
    SELECT SUM(total_rides)
    FROM daily_ridership
    WHERE year = 2012 AND working_day = 1
""").fetchone()
print('Total rides:', result[0])

Total rides: 1436146


<div dir="rtl">

**פלט צפוי**

```text
Total rides: 1436146
```

</div>

In [15]:
check(3)

OK: הפלט כצפוי.


<div dir="rtl">

התוצאה היא טאפל עם ערך אחד, ולכן המספר הוא `result[0]`.

</div>

<div dir="rtl">

**טעות נפוצה**

כאן `COUNT(*)` סופרת שורות, ולכן התשובה היא מספר ימי העבודה ולא ההשכרות בהם. כדי לחבר את ההשכרות משתמשים ב־`SUM(total_rides)`.

```python
result = c.execute("""
    SELECT COUNT(*)
    FROM daily_ridership
    WHERE year = 2012 AND working_day = 1
""").fetchone()
print('Total rides:', result[0])
```

</div>

<div dir="rtl">

**טעות נפוצה**

השורה היא טאפל גם כשיש בה ערך אחד, ולכן הדפסה שלה מראה את הסוגריים ואת הפסיק. את הערך לוקחים עם `result[0]`.

```python
result = c.execute("""
    SELECT SUM(total_rides)
    FROM daily_ridership
    WHERE year = 2012 AND working_day = 1
""").fetchone()
print('Total rides:', result)
```

</div>

<div dir="rtl">

### קבוצות והסכום הגדול ביותר

`GROUP BY` מקבץ יחד את השורות עם אותו ערך ו־`SUM` מחברת כל קבוצה. `ORDER BY ... DESC` ממיין את הקבוצות מהגדולה, ו־`LIMIT 1` משאיר את הראשונה. כאן הם מוצאים את העונה עם הכי הרבה השכרות ב־2011:

</div>

In [16]:
best = c.execute("""
    SELECT season, SUM(total_rides)
    FROM daily_ridership
    WHERE year = 2011
    GROUP BY season
    ORDER BY SUM(total_rides) DESC
    LIMIT 1
""").fetchone()
print('Season:', best[0], 'Rides:', best[1])

Season: Summer Rides: 419650


<div dir="rtl">

### משימה 4: החודש העמוס ביותר

**נסו בעצמכם.** איזה חודש בשנה היו בו הכי הרבה השכרות בכל הטבלה? ההשכרות של אותו חודש ב־2011 וב־2012 מחוברות יחד. הדפיסו את הטקסט `Busiest month:` ואחריו את הטאפל של מספר החודש וסך ההשכרות שלו, ב־`print` אחד.

</div>

<div dir="rtl">

> **רמז**
>
> מקבצים את השורות לפי `month`, כך שאותו חודש בשתי השנים הוא קבוצה אחת. ממיינים את הקבוצות לפי הסכום בסדר יורד ומשאירים אחת. השורה היא טאפל של החודש והסכום, וזה מה ש־`print` מציג.

</div>

<div dir="rtl">

**פתרון**

</div>

In [17]:
result = c.execute("""
    SELECT month, SUM(total_rides) AS total
    FROM daily_ridership
    GROUP BY month
    ORDER BY total DESC
    LIMIT 1
""").fetchone()
print('Busiest month:', result)

Busiest month: (8, 351194)


<div dir="rtl">

**פלט צפוי**

```text
Busiest month: (8, 351194)
```

</div>

In [18]:
check(4)

OK: הפלט כצפוי.


<div dir="rtl">

חודש 8, אוגוסט, הוא החודש עם הכי הרבה השכרות כששתי השנים מחוברות.

</div>

<div dir="rtl">

**טעות נפוצה**

הקבוצות ממוינות לפי מספר החודש ולא לפי הסכום, ולכן השאילתה מחזירה את החודש האחרון בשנה, דצמבר, בלי קשר להשכרות שלו. ממיינים לפי הסכום.

```python
result = c.execute("""
    SELECT month, SUM(total_rides) AS total
    FROM daily_ridership
    GROUP BY month
    ORDER BY month DESC
    LIMIT 1
""").fetchone()
print('Busiest month:', result)
```

</div>

<div dir="rtl">

**טעות נפוצה**

בלי `GROUP BY` הסכום מכסה את כל הטבלה, ולכן התוצאה היא סכום אחד של כל 731 הימים לצד החודש של שורה שרירותית. `GROUP BY month` נותן סכום אחד לכל חודש.

```python
result = c.execute("""
    SELECT month, SUM(total_rides) AS total
    FROM daily_ridership
    ORDER BY total DESC
    LIMIT 1
""").fetchone()
print('Busiest month:', result)
```

</div>

<div dir="rtl">

## ערכים ממשתני Python: פרמטרים

תוכנית שואלת בדרך כלל על ערכים שיש לה במשתנים. הערכים עוברים כ**פרמטרים** (parameters): כותבים סימן שאלה `?` ב־SQL במקום שבו שייך ערך, ונותנים את הערכים בנפרד, בטאפל, כארגומנט שני ל־`execute`. סימני השאלה מתמלאים לפי הסדר. אפשר ליצור ב־Python כמה משתנים בשורה אחת, כמו בשורה הראשונה של התיבה שלמטה: השמות משמאל מקבלים את הערכים מימין, לפי הסדר (זה [חילוץ](https://dvirross.com/he/courses/intro-data-science/glossary#unpacking), unpacking).

</div>

In [19]:
weather, year, working_day = 'Mist or cloudy', 2011, 0
query = """
    SELECT SUM(total_rides)
    FROM daily_ridership
    WHERE weather = ? AND year = ? AND working_day = ?
"""
values = (weather, year, working_day)
result = c.execute(query, values).fetchone()
print('Rides:', result[0])

Rides: 102244


<div dir="rtl">

**למה לא להכניס את הערכים לטקסט עם מחרוזת f?**

</div>

<div dir="rtl">

[מחרוזת f](https://dvirross.com/he/courses/intro-data-science/glossary#f-string) מדביקה ערך בטקסט ה־SQL, וערך שמכיל SQL משנה את השאילתה. עונה שמגיעה בצורה `Summer' OR '1'='1` סוגרת את הגרש ומוסיפה תנאי שתמיד מתקיים, ולכן השאילתה שלמטה סופרת את כל 731 הימים. זו [הזרקת SQL](https://dvirross.com/he/courses/intro-data-science/glossary#sql-injection) (SQL injection). עם `?` הערך הוא נתונים בלבד, ואותו טקסט לא מתאים לאף עונה:

</div>

In [20]:
season = "Summer' OR '1'='1"
pasted = f"""
    SELECT COUNT(*) FROM daily_ridership
    WHERE season = '{season}'
"""
print(c.execute(pasted).fetchone())
safe = """
    SELECT COUNT(*) FROM daily_ridership
    WHERE season = ?
"""
print(c.execute(safe, (season,)).fetchone())

(731,)
(0,)


<div dir="rtl">

[SQL מתוך Python](https://dvirross.com/he/courses/intro-data-science/sql-from-python#העברת-ערכים-פרמטרים) מראה זאת במלואו.

</div>

<div dir="rtl">

### משימה 5: ערכים ממשתנים

**נסו בעצמכם.** שמרו את מזג האוויר `'Clear or partly cloudy'`, את השנה 2012 ואת הערך 1 של יום עבודה בשלושה משתנים. השתמשו בהם כשלושה פרמטרים `?` כדי לקבל את מספר ההשכרות הכולל בימים האלה, והדפיסו את הטקסט `Total rides:` ואחריו את המספר.

</div>

<div dir="rtl">

> **רמז**
>
> ב־SQL יש שלושה תנאים, `weather = ?`, ‏`year = ?` ו־`working_day = ?`, שמחוברים ב־`AND`. בטאפל של הערכים המשתנים מופיעים באותו סדר כמו סימני השאלה.

</div>

<div dir="rtl">

**פתרון**

</div>

In [21]:
weather, year, working_day = 'Clear or partly cloudy', 2012, 1
query = """
    SELECT SUM(total_rides)
    FROM daily_ridership
    WHERE weather = ? AND year = ? AND working_day = ?
"""
values = (weather, year, working_day)
result = c.execute(query, values).fetchone()
print('Total rides:', result[0])

Total rides: 990444


<div dir="rtl">

**פלט צפוי**

```text
Total rides: 990444
```

</div>

In [22]:
check(5)

OK: הפלט כצפוי.


<div dir="rtl">

הטאפל `(weather, year, working_day)` עוקב אחרי הסדר של סימני השאלה.

</div>

<div dir="rtl">

**טעות נפוצה**

הערכים בסדר שונה מסימני השאלה, ולכן השנה מושווית לטקסט של מזג האוויר ומזג האוויר למספר השנה: אף שורה לא מתאימה והסכום הוא `None`. הטאפל חייב לעקוב אחרי הסדר של ה־`?`.

```python
weather, year, working_day = 'Clear or partly cloudy', 2012, 1
query = """
    SELECT SUM(total_rides)
    FROM daily_ridership
    WHERE weather = ? AND year = ? AND working_day = ?
"""
result = c.execute(query, (year, weather, working_day)).fetchone()
print('Total rides:', result[0])
```

</div>

<div dir="rtl">

## פונקציות ששולחות שאילתות

**פונקציה** נותנת שם לעבודה. השורה `def get_name(parameter):` מגדירה אותה, השורות המוזחות מתחתיה הן הגוף שלה, ו־`return` שולחת ערך חזרה למי שקרא לה. פונקציה ששולחת שאילתה מקבלת כפרמטרים את הערכים שהיא צריכה ומעבירה אותם הלאה כערכי ה־`?`. השיעור המלא נמצא ב[לולאות ופונקציות](https://dvirross.com/he/courses/intro-data-science/python-loops-and-functions#פונקציות-שם-לעבודה).

שאילתת סיכום על אפס שורות מחזירה ערך חסר: `SUM` של כלום היא `NULL`, ו־Python מציגה `NULL` כ־**[`None`](https://dvirross.com/he/courses/intro-data-science/glossary#none)**, הערך שמסמן שאין ערך. פונקציה שמחזירה מספר צריכה להחזיר 0 במקרה כזה. **[ביטוי תנאי](https://dvirross.com/he/courses/intro-data-science/glossary#conditional-expression)** (conditional expression), `a if condition else b`, הוא `a` כשהתנאי מתקיים ו־`b` אחרת, ו־`is not None` מתקיים כשיש ערך:

</div>

In [23]:
print(
    c.execute("""
        SELECT SUM(total_rides)
        FROM daily_ridership
        WHERE season = 'Summer' AND year = 2013
    """).fetchone()
)

def get_season_rides(season, year):
    query = """
        SELECT SUM(total_rides)
        FROM daily_ridership
        WHERE season = ? AND year = ?
    """
    row = c.execute(query, (season, year)).fetchone()
    return row[0] if row[0] is not None else 0

print('Summer 2011:', get_season_rides('Summer', 2011))
print('Summer 2013:', get_season_rides('Summer', 2013))

(None,)
Summer 2011: 419650
Summer 2013: 0


<div dir="rtl">

הפונקציה משתמשת בסמן `c` שנוצר למעלה. הנתונים כוללים את השנים 2011 ו־2012, ולכן לשנת 2013 אין שורות.

### משימה 6: פונקציה ששולחת שאילתה

**נסו בעצמכם.** כתבו פונקציה `get_total_rides(year, month)` שמחזירה את מספר ההשכרות הכולל של השנה והחודש שניתנו (מספר שלם מ־1 עד 12), ו־0 כשאין שורה מתאימה. הדפיסו את הטקסט `July 2012:` ואת התוצאה ליולי 2012, ואחר כך את הטקסט `January 2013:` ואת התוצאה לינואר 2013, כל אחד בשורה משלו.

</div>

<div dir="rtl">

> **רמז**
>
> בתוך הפונקציה: שאילתה עם שני פרמטרים `?`, ‏`fetchone`, וביטוי תנאי שמחזיר 0 כשהערך הראשון בשורה הוא `None`.

</div>

<div dir="rtl">

**פתרון**

</div>

In [24]:
def get_total_rides(year, month):
    query = """
        SELECT SUM(total_rides)
        FROM daily_ridership
        WHERE year = ? AND month = ?
    """
    row = c.execute(query, (year, month)).fetchone()
    return row[0] if row[0] is not None else 0

print('July 2012:', get_total_rides(2012, 7))
print('January 2013:', get_total_rides(2013, 1))

July 2012: 203607
January 2013: 0


<div dir="rtl">

**פלט צפוי**

```text
July 2012: 203607
January 2013: 0
```

</div>

In [25]:
check(6)

OK: הפלט כצפוי.


<div dir="rtl">

לשנת 2013 אין שורות, ולכן `SUM` מחזירה `None` והפונקציה מחזירה 0.

</div>

<div dir="rtl">

**טעות נפוצה**

בלי הבדיקה הפונקציה מחזירה את `row[0]` כמו שהוא, ולכן בחודש בלי שורות מי שקרא לה מקבל `None` ולא מספר.

```python
def get_total_rides(year, month):
    query = """
        SELECT SUM(total_rides)
        FROM daily_ridership
        WHERE year = ? AND month = ?
    """
    row = c.execute(query, (year, month)).fetchone()
    return row[0]

print('July 2012:', get_total_rides(2012, 7))
print('January 2013:', get_total_rides(2013, 1))
```

</div>

<div dir="rtl">

## רשימה מתוצאות שאילתה, ו־`max` עם מפתח

**[הבנת רשימה](https://dvirross.com/he/courses/intro-data-science/glossary#list-comprehension)** (list comprehension) בונה רשימה בשורה אחת: `[expression for item in rows]` נותנת את הביטוי לכל פריט. הפונקציה `max` מחזירה את הפריט הגדול ביותר ברשימה. כשהפריטים הם טאפלים, `key=` קובע במה משווים: היא מקבלת פונקציה, ו־`lambda pair: pair[1]` היא **[למדה](https://dvirross.com/he/courses/intro-data-science/glossary#lambda)** (lambda), פונקציה בשורה אחת בלי שם שמחזירה את הערך השני של טאפל. אז `max` מחזירה את הטאפל כולו שהערך השני שלו הוא הגדול ביותר.

הדוגמה שלמטה אוספת טאפלים `(season, rides)` של 2011, בעזרת הפונקציה מהסעיף הקודם. הקריאה `fetchall` מסתיימת לפני שהלולאה מתחילה, ולכן אותו סמן יכול להריץ את השאילתות שבתוכה. כל שורה של העונות היא טאפל עם ערך אחד, ולכן `season[0]` הוא הטקסט:

</div>

In [26]:
season_totals = [
    (season[0], get_season_rides(season[0], 2011))
    for season in c.execute(
        "SELECT DISTINCT season FROM daily_ridership"
    ).fetchall()
]
print(season_totals)
print(max(season_totals, key=lambda pair: pair[1]))

[('Winter', 150000), ('Spring', 347316), ('Summer', 419650), ('Fall', 326137)]
('Summer', 419650)


<div dir="rtl">

### משימה 7: רשימה ו־`max`

**נסו בעצמכם.** כתבו פונקציה עם השם והפרמטרים האלה:

```text
get_rides_by_weather_year_month(weather, year, month)
```

היא מחזירה את מספר ההשכרות הכולל בימים של מזג האוויר הזה בשנה ובחודש האלה, ו־0 כשאין שורה מתאימה. בעזרתה בנו רשימה של טאפלים `(month, rides)` עבור מזג האוויר `'Clear or partly cloudy'` ב־2012, עם טאפל אחד לכל חודש שונה בטבלה (בטבלה יש גם חודשים של 2011, ולכן בוחרים את החודשים של 2012). מצאו עם `max` את הטאפל עם הכי הרבה השכרות והדפיסו אותו.

</div>

<div dir="rtl">

> **רמז**
>
> הפונקציה דומה ל־`get_total_rides` עם תנאי אחד נוסף, `weather = ?`. הבנת הרשימה רצה על שאילתה שבוחרת את החודשים השונים של 2012, ו־`month[0]` הוא מספר החודש.

</div>

<div dir="rtl">

**פתרון**

</div>

In [27]:
def get_rides_by_weather_year_month(weather, year, month):
    query = """
        SELECT SUM(total_rides)
        FROM daily_ridership
        WHERE weather = ? AND year = ? AND month = ?
    """
    row = c.execute(query, (weather, year, month)).fetchone()
    return row[0] if row[0] is not None else 0

clear = [
    (month[0], get_rides_by_weather_year_month(
        'Clear or partly cloudy', 2012, month[0]))
    for month in c.execute("""
        SELECT DISTINCT month FROM daily_ridership
        WHERE year = 2012
    """).fetchall()
]
print(max(clear, key=lambda pair: pair[1]))

(6, 174967)


<div dir="rtl">

**פלט צפוי**

```text
(6, 174967)
```

</div>

In [28]:
check(7)

OK: הפלט כצפוי.


<div dir="rtl">

בין הימים הבהירים של 2012, ביוני יש הכי הרבה השכרות. החודש העמוס ביותר ב־2012 בכללותה הוא ספטמבר, כי שם נספרים גם ימים עם מזג אוויר אחר.

</div>

<div dir="rtl">

**טעות נפוצה**

בלי `key`, הפונקציה `max` משווה את הטאפלים מהערך הראשון שלהם, ולכן היא מחזירה את החודש עם המספר הגדול ביותר, דצמבר, ולא את החודש עם הכי הרבה השכרות.

```python
def get_rides_by_weather_year_month(weather, year, month):
    query = """
        SELECT SUM(total_rides)
        FROM daily_ridership
        WHERE weather = ? AND year = ? AND month = ?
    """
    row = c.execute(query, (weather, year, month)).fetchone()
    return row[0] if row[0] is not None else 0

clear = [
    (month[0], get_rides_by_weather_year_month(
        'Clear or partly cloudy', 2012, month[0]))
    for month in c.execute("""
        SELECT DISTINCT month FROM daily_ridership
        WHERE year = 2012
    """).fetchall()
]
print(max(clear))
```

</div>

<div dir="rtl">

## שאלות לשיחה

- איזה חלק בקוד משתנה כשמזג האוויר, השנה או החודש מגיעים ממשתנים?
- מה עלול להשתבש אם ערך מודבק בטקסט ה־SQL עם מחרוזת f?
- היכן בקוד שלכם תוצאה של שאילתה הפכה לערך רגיל של Python?

## סגירת החיבור

השיטה `close` של החיבור סוגרת את מסד הנתונים כשהעבודה נגמרת. סגירה לא שומרת שינויים (תוכנית ששינתה נתונים קוראת קודם ל־`commit`, ראו [SQL מתוך Python](https://dvirross.com/he/courses/intro-data-science/sql-from-python#שינוי-נתונים-ושמירתם)); השאילתות שלנו רק קוראות.

</div>

<div dir="rtl">

הריצו את התא שלמטה כשלב אחרון: הוא משחרר את קובץ מסד הנתונים.

</div>

In [29]:
conn.close()

<div dir="rtl">

## לזכור

- תוכנית שואלת מסד נתונים בארבעה שלבים: `connect`, ‏`cursor`, ‏`execute` ו־`fetch`. ‏`fetchall` מחזירה רשימה של טאפלים, `fetchone` טאפל אחד, ו־`fetchmany(n)` את `n` הטאפלים הבאים.
- הערכים של שורה נמצאים לפי הסדר של ה־`SELECT`, וציין לוקח אחד מהם: `row[0]`.
- מעבירים ערכים עם `?` וטאפל של ערכים, ולא מדביקים טקסט בתוך ה־SQL.
- שאילתת סיכום על אפס שורות נותנת `None`; ביטוי תנאי, כמו ב־`get_total_rides`, הופך אותה ל־0.
- ‏`max` עם `key=lambda pair: pair[1]` מחזירה את הטאפל שהערך השני שלו הוא הגדול ביותר.

</div>